In [ ]:
import sklearn
import numpy as np
import pandas as pd
import os
import re 
import sys 
import importlib
import datetime as dt

############ LOAD in custom packages ################

project_root = os.path.join(os.getcwd(), "..") # Get path of the project 
sys.path.append(project_root) # Add project root to sys.path for script usage

# Import and reload (optional) custom scripts
from scripts import paths
from scripts import preprocessing as pre
from scripts import visualization as vis
from scripts import variables
from scripts import feature_selection as fs

importlib.reload(paths)
importlib.reload(pre)
importlib.reload(vis)
importlib.reload(variables)
importlib.reload(fs)


# Define label variables
df_names = ['v1_day', 'v2_day', 'v1_week', 'v2_week']



# Filepaths
brighten_dir = paths.DATA
sub_dir = paths.SUB_DFS
results_dir = paths.RESULTS
demo_dir = paths.DEMO

In [14]:
# Create binary of depression score at the end of the time
count=0
start_depressed=np.nan
end_depressed=np.nan
last_phq9=np.nan
change_bin=np.nan
for name in ['v1_day','v2_day']: 
	phq9_end = []
	Xy = pd.read_csv(os.path.join(brighten_dir, f'{name}_trainval.csv'))
	for sub, sub_df in Xy.groupby('num_id'):
		sub_phq9 = sub_df.dropna(subset='phq9_sum')
		if len(sub_phq9) == 0:
			continue
		sub_phq9 = sub_phq9.sort_values(by='day', ascending=True)

		first_phq9 = list(sub_phq9['phq9_sum'])[0]
		if first_phq9 > 10:
			start_depressed = 1
		else:
			start_depressed = 0


		# Phq9 at ~6 weeks
		days6weeks=sub_phq9[sub_phq9['day']>38]
		days6weeks=days6weeks[days6weeks['day']<55]
		if not len(days6weeks) > 0:
			phq9_end.append([sub, first_phq9, start_depressed, last_phq9, end_depressed, change_bin]) #np.nan for last_phq9, end_depressed
			continue
		days6weeks_cols = days6weeks.dropna(how='all', axis=1)
		if not 'phq9_sum' in days6weeks_cols:
			phq9_end.append([sub, first_phq9, start_depressed, last_phq9, end_depressed, change_bin]) #np.nan for last_phq9, end_depressed
			continue

		last_phq9 = list(days6weeks['phq9_sum'])[0]
		if last_phq9 > 10:
			end_depressed = 1
		else:
			end_depressed = 0
		
		change_bin = start_depressed - end_depressed

		# Make 'Total Group' Variable:
			# 0: Start not-depressed, end not-depressed
			# 1: Start not-depressed, end depressed
			# 2: Start depressed, end not-depressed
			# 3: Start depressed, end depressed
		if start_depressed+end_depressed == 0:
			dep_group = 0
		if start_depressed == 0 and end_depressed == 1:
			dep_group = 1
		if start_depressed == 1 and end_depressed == 0:
			dep_group = 2
		if start_depressed+end_depressed == 2:
			dep_group = 3


		# if count < 3:
		# 	count+=1
		# 	print(f'Sub: {sub}')
		# 	display(days6weeks[['day','dt','phq9_sum']])
		# 	display(f'day: {list(days6weeks['day'])[0]}, phq9: {list(days6weeks['phq9_sum'])[0]}')
		

		phq9_end.append([sub, first_phq9, start_depressed, last_phq9, end_depressed, change_bin, dep_group])

	phq9_end_df = pd.DataFrame(phq9_end, columns=['num_id', 'phq9_sum_start', 'start_depressed_binary', 'phq9_sum_6wks', '6wks_depressed_binary', 'depression_change_bin', 'dep_group'])

	phq9_end_df.to_csv(os.path.join(outputs_dir, f'{name}_phq9sum_6wks.csv'), index=False)
	#display(phq9_end_df)
	print(f'\n\nSaved phq9_end_df to {name}_phq9sum_6wks.csv')
	print(phq9_end_df['start_depressed_binary'].value_counts())
	print(phq9_end_df['6wks_depressed_binary'].value_counts())
	print(phq9_end_df['depression_change_bin'].value_counts())
	print(phq9_end_df['dep_group'].value_counts())



print('Run on:', dt.datetime.today().strftime('%a %d %b %Y, %I:%M%p'))



Saved phq9_end_df to v1_day_phq9sum_6wks.csv
start_depressed_binary
0    106
1     73
Name: count, dtype: int64
6wks_depressed_binary
0    109
1     70
Name: count, dtype: int64
depression_change_bin
 0    133
-1     23
 1     23
Name: count, dtype: int64
dep_group
0.0    72
3.0    39
2.0    21
1.0    19
Name: count, dtype: int64


Saved phq9_end_df to v2_day_phq9sum_6wks.csv
start_depressed_binary
1    84
0    62
Name: count, dtype: int64
6wks_depressed_binary
1    75
0    71
Name: count, dtype: int64
depression_change_bin
 0    105
 1     26
-1     15
Name: count, dtype: int64
dep_group
3.0    28
0.0    26
2.0    14
1.0     5
Name: count, dtype: int64
Run on: Thu 28 May 2026, 10:47AM


In [15]:
v1_phq9_end_df=pd.read_csv(os.path.join(outputs_dir, f'v1_day_phq9sum_6wks.csv'))
v2_phq9_end_df=pd.read_csv(os.path.join(outputs_dir, f'v2_day_phq9sum_6wks.csv'))
phq9_end_df = pd.concat([v1_phq9_end_df, v2_phq9_end_df], axis=0)
phq9_end_df = phq9_end_df.loc[:, ~phq9_end_df.columns.str.contains('Unnamed')]
display(phq9_end_df)

phq9_end_df.to_csv(os.path.join(outputs_dir, f'phq9sum_6wks.csv'), index=False)

,num_id,phq9_sum_start,start_depressed_binary,phq9_sum_6wks,6wks_depressed_binary,depression_change_bin,dep_group
0,13.0,9.0,0,15.0,1,-1,1.0
1,14.0,10.0,0,12.0,1,-1,1.0
2,17.0,18.0,1,12.0,1,-1,NaN
3,29.0,12.0,1,7.0,0,1,2.0
4,44.0,9.0,0,7.0,0,0,0.0
...,...,...,...,...,...,...,...
141,1081.0,4.0,0,7.0,0,0,0.0
142,1096.0,19.0,1,20.0,1,0,3.0
143,1100.0,9.0,0,20.0,1,0,NaN
144,1101.0,4.0,0,20.0,1,0,NaN


In [16]:
# Merge in 
for name in df_names:
    df = pd.read_csv(os.path.join(brighten_dir, f'{name}_trainval_transformed.csv'), low_memory=False)

    if 'v1' in name:
        phq9_change = pd.read_csv(os.path.join(outputs_dir, 'v1_day_phq9sum_6wks.csv')).rename(columns={'week':'ending_week'})
    else:
        phq9_change = pd.read_csv(os.path.join(outputs_dir, 'v2_day_phq9sum_6wks.csv')).rename(columns={'week':'ending_week'})

    phq9_change=phq9_change.drop(columns=[col for col in phq9_change.columns if 'Unnamed' in col])
    merge_df = df.merge(phq9_change, on=['num_id'],how='outer')
    merge_df.to_csv(os.path.join(brighten_dir, f'{name}_trainval_transformed_outcomes.csv'), index=False)

print('Run on:', dt.datetime.today().strftime('%a %d %b %Y, %I:%M%p'))


Run on: Thu 28 May 2026, 10:49AM
